In [122]:
import argparse
import json
import re
from pathlib import Path
import sys

from random import choices 
from collections import defaultdict

In [123]:
# open file, get text 
IBpath = "omens_working"
IB_text_path = Path(IBpath)

if not IB_text_path.exists():
    raise FileExistsError

IB_text = IB_text_path.read_text()
#text

IB_text

"<!--Omen 1: 2-2-2-->\n{{RTL|◉◉ ◉◉ ◉◉}}\n\n<!--Divination 1-->\n{{RTL|𐱅𐰤⸱𐰾𐰃⸱𐰢𐰤⸱𐰖𐰺𐰣⸱𐰚𐰃𐰲𐰀⸱𐰞𐱃𐰆𐰣⸱𐰇𐱁𐰏𐰃𐰤⸱𐰇𐰔𐰀⸱𐰆𐰞𐰆𐰺𐰰𐰣⸱𐰢𐰭𐰃𐰠𐰘𐰇𐱁⸱𐰢𐰤⸱𐰨𐰀⸱𐰋𐰃𐰠𐰃𐰭𐰠𐱁⸱𐰓𐰏𐰇⸱𐰆𐰞⸱}}\n<!--Transliteration : t²n² s²i mn² y¹r¹n¹ k²iča l¹t¹on¹ öšg²in² öza ol¹or¹o͡pn¹ mŋil²y²öš mn² n͜ča b²il²iŋl²š d²g²ö ol¹-->\n<!--Transcription : tän·si män. yarın kiçä altun örgin üzä olurupan mäŋiläyür män. ança biliŋlär: ädgü ol.-->\n<!--Prognostication : good-->\n\n<br/>\n<!--Omen 2: 4-4-4-->\n{{RTL|◉◉◉◉ ◉◉◉◉ ◉◉◉◉}}\n\n<!--Divination 2-->\n{{RTL|𐰀𐰞𐰀⸱𐱃𐰞𐰍⸱𐰖𐰆𐰞⸱𐱅𐰭𐱁𐰃⸱𐰢𐰤⸱𐰖𐰺𐰣⸱𐰚𐰃𐰲𐰀⸱𐰾𐰇𐱁⸱𐰢𐰤⸱𐰆𐱃𐰺𐰆⸱𐰚𐰃⸱𐰖𐰞𐰍⸱𐰚𐰃𐰾𐰃⸱𐰆𐰍𐰞𐰃𐰣⸱𐰽𐰆𐰸𐰆𐰽𐰢𐰃𐰾⸱𐰚𐰃𐰾𐰃⸱𐰴𐰆𐰺𐰴𐰢𐰃𐰾⸱𐰴𐰆𐰺𐰴𐰢𐰀⸱𐱅𐰃𐰢𐰾⸱𐰴𐰆𐱃⸱𐰋𐰃𐱁𐰏𐰘⸱𐰢𐰤⸱𐱅𐰃𐰢𐰃𐰾⸱𐰨𐰀⸱𐰋𐰃𐰠𐰃𐰭⸱𐰓𐰏𐰇⸱𐰆𐰞⸱}}\n<!--Transliteration : al¹a t¹l¹g¹ y¹ol¹ t²ŋši mn² y¹r¹n¹ k²iča s²öš mn² ot¹r¹o k²i y¹l¹g¹ k²is²i og¹l¹in¹ s¹oo͜kos¹mis² k²is²i k¹or¹k¹mis² k¹or¹k¹ma t²ims² k¹ot¹ b²išg²y² mn² t²imis² n͜ča b²il²iŋ d²g²ö ol¹-->\n<!--Transcription : ala atlıg yol täŋri män, yarın kiçä äşür män. utru eki yalıg kişi oglın sookuşmiş. kişi korkmiş. 'korkma' timiş, 'kut birgäy män' timiş. ança bil

In [132]:
# Parsing the source text
## Component patterns
omen_label = r"<!--Omen\s*(?P<omen_number>\d+):\s*(?P<number_pattern>.*?)-->"
dot_pattern = r"\{\{RTL\|(?P<dot_pattern>.*?)\}\}"

divination_label = r"<!--Divination\s*(?P<divination_number>\d+)\s*-->"
text_match = r"\{\{RTL\|(?P<omen_text>.*?)\}\}"
transliteration_pattern = r"<!--Transliteration\s*:\s*(?P<transliteration>.*?)-->"
transcript_pattern = r"<!--Transcription\s*:\s*(?P<transcription>.*?)-->"

prognostication_pattern = r"(<!--Prognostication\s*:\s*(?P<prognostication>.*?)-->)?" 

omen_pattern = r"\s*".join([omen_label, dot_pattern, divination_label, text_match,
                            transliteration_pattern, transcript_pattern, 
                            prognostication_pattern])
omen_regex = re.compile(omen_pattern)

# Parse the file into a list of dictionaries.
omen_list = [match.groupdict() for match in omen_regex.finditer(IB_text)]
omen_list

patterns = set()
for pattern in [o['number_pattern'] for o in omen_list]:
    patterns.add(''.join(pattern.split("-")))

def ppatterns():
    out = set()
    for a in '1234':
        for b in '1234':
            for c in '1234':
                out.add(a+b+c)
    return out

ppatterns() - patterns

{'124', '311'}

In [119]:
class Omen:
    def __init__(self, data:dict):
        omen_number, divination_number = map(int, (data['omen_number'], data['divination_number']))
        if omen_number != divination_number:
            raise ValueError(f"Divination number mismatch in omen number {omen_number}")
        self.number = omen_number
        
        self.number_pattern = ''.join(data['number_pattern'].split('-'))
        self.dot_pattern = data['dot_pattern'].split()

        self.text = data['omen_text']
        self.transliteration = data['transliteration']
        self.transcription = data['transcription']
        self.prognostication = data['prognostication']

    def __iter__(self):
        yield self 

    def __repr__(self):
        return f"Omen {self.number}"
    
    def split_text(self):
        return [word for word in self.text.split("⸱") if word]
    
    def split_transcription(self):
        words = re.split(r"[·\s]+", self.transcription)
        return [re.sub(r"[,:;'!\?\!\.]", r"", word) for word in words if word]

    def word_table(self, text=True, transliteration=True, transcription=False):
        columns = list()
        if text:
            alpha = self.split_text()
            columns.append(alpha)
        if transliteration:
            translit = self.transliteration.split() 
            assert len(translit) == len(alpha) 
            columns.append(translit)
        if transcription:
            transcript = self.split_transcription()
            assert len(transcript) == len(alpha)
            columns.append(transcript)
        return list(zip(*columns))
        
class OmenList: 
    def __init__(self, omens):
        number_index = dict()
        pattern_index = dict()
        for this in omens:
            number_index[this.number] = this 

            pattern = this.number_pattern
            record = pattern_index.get(pattern, None)
            if record is None:
                pattern_index[pattern] = [this] 
            else:
                pattern_index[pattern].append(this) 
        self.number_index = number_index
        self.pattern_index = pattern_index

    def get_omen_by_number(self, number):
        if 1 <= number <= 65:
            return self.number_index[number]
        else:
            raise ValueError(f"Omen number out of range: {number}")
        
    def get_omens_by_pattern(self, pattern:str):
        if not re.match(r"[1234\*][1234\*][1234\*]", pattern):
            raise ValueError(f"Bad value in pattern: {pattern}")
        
        p1, p2, p3 = pattern
        if p1 == "*":
            p1 = None
        if p2 == "*":
            p2 = None 
        if p3 == "*":
            p3 = None 

        if all((p1,p2,p3)):
            try:
                omens = self.pattern_index[pattern]
            except KeyError:
                raise ValueError(f"Bad pattern: {pattern}")
            else:
                return omens 
        else:
            patterns = self.pattern_index.keys()
            if p1:
                patterns = filter(lambda s:s.startswith(p1), patterns)
            if p2:
                patterns = filter(lambda s:s[1] == p2, patterns)
            if p3:
                patterns = filter(lambda s:s[2] == p3, patterns)
            omens = list()
            index = self.pattern_index 
            for pattern in patterns:
                omens.extend(index[pattern])
            return omens 

Omens = OmenList(Omen(d) for d in omen_list)
Omens.get_omens_by_pattern("132")

[Omen 45]

In [ ]:
class OmenContainer:
    def __init__(self, omens):
        self.omen_list = tuple(omens)

    def by_number(self, number):
        if 1 <= number <= 65:
            return self.omen_list


    def __init__(self, omens):
        self.omen_list = list(omens)

    def __getitem__(self, i):
        if isinstance(i, int):
            if i < 1:
                raise IndexError("Omen index starts at 1")
            elif i > 65:
                raise IndexError(f"Omen index out of range: {i}")
            else:
                omen = self.omen_list[i - 1]
                
        elif isinstance(i, tuple):
            return [self[j] for j in i]
        
    def __iter__(self):
        yield from self.omen_list
    
    def label_by(self, attribute_name):
        data = dict()
        for omen in self:
            index = getattr(omen, attribute_name)
            record = data.get(index, None)
            if record is None:
                data[index] = [omen] 
            else:
                data[index].append(omen)
        return data 
    
            

Omens = OmenList(Omen(omen) for omen in omen_list)
#Omens.pattern_index

Omens[1]


Omen 1

In [140]:
index

NameError: name 'index' is not defined

In [58]:

from collections import OrderedDict

o = OrderedDict(zip('123', 'abc'))
help(o)

Help on OrderedDict object:

class OrderedDict(builtins.dict)
 |  Dictionary that remembers insertion order
 |  
 |  Method resolution order:
 |      OrderedDict
 |      builtins.dict
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __delitem__(self, key, /)
 |      Delete self[key].
 |  
 |  __eq__(self, value, /)
 |      Return self==value.
 |  
 |  __ge__(self, value, /)
 |      Return self>=value.
 |  
 |  __gt__(self, value, /)
 |      Return self>value.
 |  
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __ior__(self, value, /)
 |      Return self|=value.
 |  
 |  __iter__(self, /)
 |      Implement iter(self).
 |  
 |  __le__(self, value, /)
 |      Return self<=value.
 |  
 |  __lt__(self, value, /)
 |      Return self<value.
 |  
 |  __ne__(self, value, /)
 |      Return self!=value.
 |  
 |  __or__(self, value, /)
 |      Return self|value.
 |  
 |  __reduce__(...)
 |      Return state info

In [39]:

def by_pattern(*pattern):
    try:
        match len(pattern):
            case 1:
                pattern = pattern[0]
            case 3:
                pass
            case _:
                raise ValueError
        p1, p2, p3 = pattern 
    except ValueError:
        raise ValueError(f"Invalid pattern shape. Expects three values, got: {len(pattern)}")
    ...

    

for o in Omens:
    print(o, o.number_pattern)


Omen 1 222
Omen 2 444
Omen 3 333
Omen 4 111
Omen 5 242
Omen 6 122
Omen 7 212
Omen 8 123
Omen 9 321
Omen 10 243
Omen 11 443
Omen 12 343
Omen 13 342
Omen 14 234
Omen 15 141
Omen 16 214
Omen 17 233
Omen 18 241
Omen 19 413
Omen 20 223
Omen 21 331
Omen 22 112
Omen 23 442
Omen 24 313
Omen 25 313
Omen 26 421
Omen 27 422
Omen 28 211
Omen 29 432
Omen 30 423
Omen 31 144
Omen 32 113
Omen 33 424
Omen 34 244
Omen 35 434
Omen 36 411
Omen 37 134
Omen 38 314
Omen 39 224
Omen 40 441
Omen 41 324
Omen 42 414
Omen 43 334
Omen 44 142
Omen 45 132
Omen 46 133
Omen 47 114
Omen 48 344
Omen 49 341
Omen 50 143
Omen 51 433
Omen 52 312
Omen 53 232
Omen 54 131
Omen 55 412
Omen 56 231
Omen 57 221
Omen 58 322
Omen 59 323
Omen 60 431
Omen 61 341
Omen 62 213
Omen 63 121
Omen 64 341
Omen 65 332


In [632]:
re.sub(r"[,:;'!\?\!\.]", "", "'olamm'!ng'")


'olammng'

In [652]:
o = Omens[57]
o.transcription

ts = re.compile(r"[·,'!:;\?\.\s]+")
ts.split(o.transcription),tr

#tr = o.transcription
#tr
#re.split(
# for index, omen in enumerate(Omens):
#     try:
#         print(omen.word_table())
#     except AssertionError:
#         print(f"Omen({index + 1})")

Omens[3].word_table()

[('𐰇𐰼𐰇𐰭', 'ör²öŋ'),
 ('𐰾𐱁𐰃', 's²ši'),
 ('𐱃𐰆𐰍𐰣', 't¹og¹n¹'),
 ('𐰴𐰆𐰽', 'k¹os¹'),
 ('𐰢𐰤', 'mn²'),
 ('𐰲𐰃𐰣𐱃𐰣', 'čin¹t¹n¹'),
 ('𐰃𐰍𐰲', 'ig¹č'),
 ('𐰇𐰔𐰀', 'öza'),
 ('𐰆𐰞𐰆𐰺𐰆𐰰𐰣', 'ol¹or¹oo͡pn¹'),
 ('𐰢𐰭𐰃𐰠𐰘𐰇𐱁', 'mŋil²y²öš'),
 ('𐰢𐰤', 'mn²'),
 ('𐰨𐰀', 'n͜ča'),
 ('𐰋𐰃𐰠𐰃𐰭𐰠𐱁', 'b²il²iŋl²š')]

In [660]:

def segment_transliteration_word(word):
   out = list()
   memo = None
   for char in word:
      if char in '²¹':
         out.append(out.pop() + char)
      elif char in ['͜', '͡']:
         memo = out.pop() + char 
      elif memo:
         out.append(memo + char)
         memo = None 
      else:
         out.append(char)
   return out


d = defaultdict(set)

for script_word, translit_word in Omens[0].word_table():
   for a, t in zip(script_word, segment_transliteration_word(translit_word)):
      d[a].add(t)

charmap = defaultdict(set)

for omen in Omens:
   for script_word, translit_word in omen.word_table():
      for a, t in zip(script_word, segment_transliteration_word(translit_word)):
         charmap[a].add(t)

charmap


defaultdict(set,
            {'𐱅': {'t²'},
             '𐰤': {'n²'},
             '𐰾': {'s²'},
             '𐰃': {'i', 'o'},
             '𐰢': {'m'},
             '𐰖': {'y¹'},
             '𐰺': {'r¹'},
             '𐰣': {'n¹'},
             '𐰚': {'k²', 'o'},
             '𐰲': {'č'},
             '𐰀': {'a'},
             '𐰞': {'l¹'},
             '𐱃': {'t¹'},
             '𐰆': {'o', 's²'},
             '𐰇': {'ö'},
             '𐱁': {'š'},
             '𐰏': {'g²'},
             '𐰔': {'z'},
             '𐰰': {'o͡p'},
             '𐰭': {'ŋ'},
             '𐰠': {'l²'},
             '𐰘': {'y²'},
             '𐰨': {'n͜č'},
             '𐰋': {'b²'},
             '𐰓': {'d²'},
             '𐰍': {'g¹'},
             '𐰽': {'s¹'},
             '𐰸': {'o͜k'},
             '𐰴': {'k¹'},
             '𐰑': {'d¹'},
             '𐰯': {'p'},
             '𐰜': {'ö͜k'},
             '𐰦': {'n͜t'},
             '𐰼': {'r²'},
             '𐰉': {'b¹'},
             '𐰪': {'n͡y'},
             '𐱇': {'o͜t'}})

In [ ]:
transliteration_mismatch = list()

for omen in Omens:
    word_length = len(omen.split_text())
    if len(omen.transliteration.split()) != word_length:
        transliteration_mismatch.append(omen) 

assert len(transliteration_mismatch) == 0
# Transliterations lines up word-for-word with OT script. 

transcription_mismatch = list()

for omen in Omens:
    word_length = len(omen.split_text())
    if len(omen.transcription.split()) != word_length:
        transcription_mismatch.append(omen) 

transcription_mismatch
# Some of the transcriptions have different word boundaries compared to the 
# original text and transcriptions. 

[Omen 1, Omen 3, Omen 18, Omen 29, Omen 40, Omen 44, Omen 52, Omen 55, Omen 58]

In [586]:
# Print out the omens with a mismatch so I can figure out how to make them match manually. 

def print_omen(omen:Omen):
    """Print "(word count) line" for each of omen text, transliteration, transcription.

    Reverse word order of right-to-left (RTL) Old Turkish script and add space between the words
    so it lines up better with left-to-right transliterations and transcription text. This makes
    visual alignment a little easier. Note that letter order within a word is still RTL."""

    words = omen.split_text()
    print(f"({len(words)})\t{'  '.join(reversed(words))}")
    transliteration = omen.transliteration 
    print(f"({len(transliteration.split())})\t{transliteration}")
    transcription = omen.transcription 
    print(f"({len(transcription.split())})\t{transcription}")

def print_mismatched_omens(mismatches):
    """Output looks like:
Omen 1
(15)	𐰆𐰞  𐰓𐰏𐰇  𐰋𐰃𐰠𐰃𐰭𐰠𐱁  𐰨𐰀  𐰢𐰤  𐰢𐰭𐰃𐰠𐰘𐰇𐱁  𐰆𐰞𐰆𐰺𐰰𐰣  𐰇𐰔𐰀  𐰇𐱁𐰏𐰃𐰤  𐰞𐱃𐰆𐰣  𐰚𐰃𐰲𐰀  𐰖𐰺𐰣  𐰢𐰤  𐰾𐰃  𐱅𐰤
(15)	t²n² s²i mn² y¹r¹n¹ k²iča l¹t¹on¹ öšg²in² öza ol¹or¹o͡pn¹ mŋil²y²öš mn² n͜ča b²il²iŋl²š d²g²ö ol¹
(14)	tänsi män. yarın kiçä altun örgin üzä olurupan mäŋiläyür män. ança biliŋlär: ädgü ol.

Omen 3
etc. for each omen in mismatches"""
    for omen in mismatches:
        print(repr(omen))
        print_omen(omen)
        print()

# Cell output from this function call below got copied to alignment.txt.
# That file has been edited, so uncomment this and run if you want to see the orignal output.

#print_mismatched_omens(transcription_mismatch)


In [66]:
# transliteration characters 
tl_char = set()

e = list()
for omen in omen_data.values():
    tl_char.update(omen['transliteration'])

sorted(tl_char)[-2:]


['͜', '͡']

In [ ]:

def segment_transliteration_word(word):
   out = list()
   memo = None
   for char in word:
      if char in '²¹':
         out.append(out.pop() + char)
      elif char in ['͜', '͡']:
         memo = out.pop() + char 
      elif memo:
         out.append(memo + char)
         memo = None 
      else:
         out.append(char)
   return out 

tl = omen_data[1]['transliteration']
tx = omen_data[1]['text']
tr = omen_data[1]['transcription']

list(zip(tl.split(), tx.split('⸱')))



[('t²n²', '𐱅𐰤'),
 ('s²i', '𐰾𐰃'),
 ('mn²', '𐰢𐰤'),
 ('y¹r¹n¹', '𐰖𐰺𐰣'),
 ('k²iča', '𐰚𐰃𐰲𐰀'),
 ('l¹t¹on¹', '𐰞𐱃𐰆𐰣'),
 ('öšg²in²', '𐰇𐱁𐰏𐰃𐰤'),
 ('öza', '𐰇𐰔𐰀'),
 ('ol¹or¹o͡pn¹', '𐰆𐰞𐰆𐰺𐰰𐰣'),
 ('mŋil²y²öš', '𐰢𐰭𐰃𐰠𐰘𐰇𐱁'),
 ('mn²', '𐰢𐰤'),
 ('n͜ča', '𐰨𐰀'),
 ('b²il²iŋl²š', '𐰋𐰃𐰠𐰃𐰭𐰠𐱁'),
 ('d²g²ö', '𐰓𐰏𐰇'),
 ('ol¹', '𐰆𐰞')]

In [106]:
start = 0

chunks = list()

for match in omen_re.finditer(t):
    end = match.start()
    chunks.append(t[start:end])
    start = end 

omens = list()

for omen in chunks[1:]:
    data = omen_re.match(omen).groupdict() 
    omens.append(data)

omens


[{'o_number': '1',
  'o_pattern': '2-2-2',
  'dot_pattern': '◉◉ ◉◉ ◉◉',
  'div_number': '1',
  'text': '𐱅𐰤⸱𐰾𐰃⸱𐰢𐰤⸱𐰖𐰺𐰣⸱𐰚𐰃𐰲𐰀⸱𐰞𐱃𐰆𐰣⸱𐰇𐱁𐰏𐰃𐰤⸱𐰇𐰔𐰀⸱𐰆𐰞𐰆𐰺𐰰𐰣⸱𐰢𐰭𐰃𐰠𐰘𐰇𐱁⸱𐰢𐰤⸱𐰨𐰀⸱𐰋𐰃𐰠𐰃𐰭𐰠𐱁⸱𐰓𐰏𐰇⸱𐰆𐰞⸱',
  'transliteration': 't²n² s²i mn² y¹r¹n¹ k²iča l¹t¹on¹ öšg²in² öza ol¹or¹o͡pn¹ mŋil²y²öš mn² n͜ča b²il²iŋl²š d²g²ö ol¹',
  'transcription': 'tänsi män. yarın kiçä altun örgin üzä olurupan mäŋiläyür män. ança biliŋlär: ädgü ol.'},
 {'o_number': '2',
  'o_pattern': '4-4-4',
  'dot_pattern': '◉◉◉◉ ◉◉◉◉ ◉◉◉◉',
  'div_number': '2',
  'text': '𐰀𐰞𐰀⸱𐱃𐰞𐰍⸱𐰖𐰆𐰞⸱𐱅𐰭𐱁𐰃⸱𐰢𐰤⸱𐰖𐰺𐰣⸱𐰚𐰃𐰲𐰀⸱𐰾𐰇𐱁⸱𐰢𐰤⸱𐰆𐱃𐰺𐰆⸱𐰚𐰃⸱𐰖𐰞𐰍⸱𐰚𐰃𐰾𐰃⸱𐰆𐰍𐰞𐰃𐰣⸱𐰽𐰆𐰸𐰆𐰽𐰢𐰃𐰾⸱𐰚𐰃𐰾𐰃⸱𐰴𐰆𐰺𐰴𐰢𐰃𐰾⸱𐰴𐰆𐰺𐰴𐰢𐰀⸱𐱅𐰃𐰢𐰾⸱𐰴𐰆𐱃⸱𐰋𐰃𐱁𐰏𐰘⸱𐰢𐰤⸱𐱅𐰃𐰢𐰃𐰾⸱𐰨𐰀⸱𐰋𐰃𐰠𐰃𐰭⸱𐰓𐰏𐰇⸱𐰆𐰞⸱',
  'transliteration': 'al¹a t¹l¹g¹ y¹ol¹ t²ŋši mn² y¹r¹n¹ k²iča s²öš mn² ot¹r¹o k²i y¹l¹g¹ k²is²i og¹l¹in¹ s¹oo͜kos¹mis² k²is²i k¹or¹k¹mis² k¹or¹k¹ma t²ims² k¹ot¹ b²išg²y² mn² t²imis² n͜ča b²il²iŋ d²g²ö ol¹',
  'transcription': "ala atlıg yol täŋri män, yarın kiçä äşür män. utru eki yalıg kişi oglın sookuşmiş. kişi korkmiş. 'korkma' timiş, 'kut birgäy

In [172]:
# Irk Bitig has 65 omens. Each one is mapped to a pattern of three numbers.
# 4-sided dice rolls
# This gives 4 * 4 * 4 = 64 possible patterns.
# At least one pattern must be duplicated, and actually some patterns are missing.

all_patterns_text = {omen.number_pattern for omen in Omens}
all_patterns_text

all_patterns_possible = set()

for a in '1234':
    for b in '1234':
        for c in '1234':
            all_patterns_possible.add(a + b + c)

missing_patterns = all_patterns_possible - all_patterns_text 
missing_patterns

#'124', '311'

{'124', '311'}

In [549]:
class FateError(Exception):
    ...

def roll(tries=100):
    """Roll virtual dice for your omen!"""
    rolls = tries 
    while rolls:
        out = ''.join(choices("1234", k=3))
        if out in ('123', '311'):
            rolls -= 1
        else:
            return out
    else:
        raise FateError(
            f"After {tries} tries, no omen could be found. ⸱𐰞𐰴𐰆⸱𐰚𐰤𐱅𐰇⸱𐰇𐰠𐰇𐰏𐰃⸱𐱁𐰚𐰠𐰃𐰏⸱𐰆𐰞⸱")

In [ ]:
# load html from another online version of Irk Bitig
# HTML is a mess, but this one has translations and linguistics details about each word.

import requests
from bs4 import BeautifulSoup as bs


p = requests.get("https://vatec2.fkidg1.uni-frankfurt.de/vatecasp/Irk_Bitig.htm")

In [ ]:
from itertools import pairwise


pt = p.text 
##tspat = re.compile(r"Textabschnitt")

##tspat.findall(pt)
##pt.splitlines()

pp = re.compile(r"<A NAME=\"Irk (\d+)\">Textabschnitt</A>")
indices = [match.start() for match in pp.finditer(pt)]
#for m in pp.finditer(pt):
#    indices.append(m.start())

chunks = [pt[start:end] for start, end in pairwise(indices)]
#for start, end in pairwise(indices):
#    chunks.append(pt[start:end])
chunks

br = re.compile(r"<BR>", re.IGNORECASE)

c = chunks[0]
re.sub(r"(<BR>)", 'XX', c,flags=re.IGNORECASE)

Help on built-in function sub:

sub(repl, string, count=0) method of re.Pattern instance
    Return the string obtained by replacing the leftmost non-overlapping occurrences of pattern in string by the replacement repl.



In [77]:
br = re.compile(r"<BR>+", re.IGNORECASE)

with open('wacky_html', 'w') as file:
    for n, chunk in enumerate(chunks):
        chunk = br.sub('', chunk)
        file.write(f'<IRK index={n}>\n')
        for line in chunk.splitlines():
            if not line:
                continue
            file.write('\t')
            file.write(line)
            file.write('\n')
        file.write('</IRK>\n')



In [ ]:
with open('wacky_html') as file:
    bp = bs(file)

i = bp.find_all('irk')[3]


<irk index="3">
<a name="Irk 04">Textabschnitt</a>:<font color="000080" size="+2"><a name="142777">Irk</a> <a name="142778">04</a> 
</font>
<div align="CENTER"><b><font color="aaaaaa" size="+1">Referenz:</font><font color="000000" size="+2"><a name="142779">Irk</a> <a name="142780">04:</a> <a name="142781">08(l):01</a> </font></b></div>
<font size="+1">
<table align="left" border="1">
<tr><td><font color="aaaaaa">Transliteration</font></td><td><font color="FF0080"><a name="142782">&lt;ws=graph&gt;Irk4&lt;/ws&gt;</a></font></td></tr>
<tr><td><font color="aaaaaa">genaue Transkription</font></td><td><font color="00C080"> </font></td></tr>
</table><br clear="all"/></font>
<div align="CENTER"><b><font color="aaaaaa" size="+1">Referenz:</font><font color="000000" size="+2"><a name="142783">Irk</a> <a name="142784">04:</a> <a name="142785">08(l):02</a> </font></b></div>
<font size="+1">
<table align="left" border="1">
<tr><td><font color="aaaaaa">Transliteration</font></td><td><font color="FF

In [57]:

i

NameError: name 'i' is not defined